In [ ]:
%cd ../
%load_ext autoreload
%autoreload 2


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# Load the raw measurements data
df = pd.read_parquet('data/raw_measurements.parquet')
print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


In [ ]:
# Check data types and structure
print("Data info:")
df.info()
print(f"\nNumber of unique stations: {df['station_id'].nunique()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")


In [ ]:
# Ensure timestamp is datetime type
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort by station_id and timestamp to ensure proper ordering
df = df.sort_values(['station_id', 'timestamp'])

print(f"Data sorted by station_id and timestamp")
print(f"First few rows:")
df.head(10)


In [ ]:
# Create hourly timestamp by flooring to the hour
df['hour'] = df['timestamp'].dt.floor('H')

# Get all columns except timestamp and hour for aggregation
value_columns = [col for col in df.columns if col not in ['timestamp', 'hour', 'station_id']]

print(f"Columns to aggregate: {value_columns}")


In [ ]:
# Aggregate to hourly resolution by taking the mean
# Group by station_id and hour, then calculate mean for all value columns
hourly_df = df.groupby(['station_id', 'hour'])[value_columns].mean().reset_index()

# Rename 'hour' back to 'timestamp' for consistency
hourly_df = hourly_df.rename(columns={'hour': 'timestamp'})

print(f"Hourly aggregated data shape: {hourly_df.shape}")
print(f"Original data shape: {df.shape}")
print(f"Reduction ratio: {df.shape[0] / hourly_df.shape[0]:.2f}x")
print(f"\nFirst few rows of aggregated data:")
hourly_df.head(10)


In [ ]:
# Verify the aggregation
print("Aggregated data info:")
hourly_df.info()
print(f"\nDate range: {hourly_df['timestamp'].min()} to {hourly_df['timestamp'].max()}")
print(f"Number of unique stations: {hourly_df['station_id'].nunique()}")


In [ ]:
# Save the hourly aggregated data
output_path = 'data/hourly_measurements.parquet'
hourly_df.to_parquet(output_path, index=False)
print(f"Hourly aggregated data saved to: {output_path}")


In [1]:
# Verify the saved file
verification_df = pd.read_parquet(output_path)
print(f"Verification - loaded data shape: {verification_df.shape}")
print(f"Columns match: {list(verification_df.columns) == list(hourly_df.columns)}")
print(f"\nSample of loaded data:")
verification_df.head()


NameError: name 'pd' is not defined